# External URLs — Data Cleaning

This notebook addresses the data quality issues identified
during the exploratory analysis of external_urls.json.

The main objectives are to:
- Remove exact duplicates.
- Validate identifiers and URL syntax.
- Investigate malformed URLs.
- Check referential integrity with the posts dataset.
- Evaluate cross-dataset recovery opportunities.
- Derive useful URL-related features.
- Export a validated cleaned dataset.

Original records are preserved for reproducibility.

In [1]:
from pathlib import Path
import pandas as pd
from urllib.parse import urlsplit

# Define dataset path
DATA_DIR = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data\Project_Data"
)

EXTERNAL_URLS_PATH = DATA_DIR / "external_urls.json"

if not EXTERNAL_URLS_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {EXTERNAL_URLS_PATH}"
    )

# Load original dataset
df_urls_raw = pd.read_json(EXTERNAL_URLS_PATH)

# Create working copy
df_urls_clean = df_urls_raw.copy()

print("Original dataset shape:", df_urls_raw.shape)

print("\nColumns and data types:")
display(df_urls_raw.dtypes.to_frame("dtype"))

print("\nDataset preview:")
display(df_urls_raw.head())

Original dataset shape: (252877, 4)

Columns and data types:


,dtype
id_ext_url,int64
url,str
truth_id,int64
url_id,int64



Dataset preview:


,id_ext_url,url,truth_id,url_id
0,1780,https://www.supremecourt.gov/opinions/21pdf/19...,3207,1780
1,1780,https://www.supremecourt.gov/opinions/21pdf/19...,41909,1780
2,1780,https://www.supremecourt.gov/opinions/21pdf/19...,80108,1780
3,1780,https://www.supremecourt.gov/opinions/21pdf/19...,251504,1780
4,1780,https://www.supremecourt.gov/opinions/21pdf/19...,272831,1780


## Duplicate Removal

The exploratory analysis identified five exact duplicate
records, corresponding to repeated post–URL associations.

These duplicates can be safely removed because they do
not represent additional unique relationships.

The original dataset is preserved.

In [2]:
# Count exact duplicates
exact_duplicates = df_urls_clean.duplicated().sum()

# Count duplicate post-URL associations
duplicate_pairs = df_urls_clean.duplicated(
    subset=["truth_id", "url_id"]
).sum()

print("Exact duplicates:", exact_duplicates)
print("Duplicate post-URL pairs:", duplicate_pairs)

# Remove exact duplicates
df_urls_clean = (
    df_urls_clean
    .drop_duplicates()
    .copy()
)

# Validate cleaning
print("\nOriginal rows:", len(df_urls_raw))
print("Cleaned rows:", len(df_urls_clean))

print(
    "Remaining duplicates:",
    df_urls_clean.duplicated().sum()
)

assert df_urls_clean.duplicated().sum() == 0

Exact duplicates: 5
Duplicate post-URL pairs: 5

Original rows: 252877
Cleaned rows: 252872
Remaining duplicates: 0


## Identifier and Structural Validation

We validate the identifiers used to represent the relationships
between posts and external URLs.

The main objectives are to:
- Check for missing, invalid, or non-positive identifiers.
- Verify consistency between id_ext_url and url_id.
- Check the uniqueness of post-URL relationships.
- Identify potential structural inconsistencies.

No identifiers are modified or removed without evidence of
an actual data quality issue.

In [3]:
# Define identifier columns
id_columns = [
    "truth_id",
    "id_ext_url",
    "url_id"
]

# Convert identifiers to nullable integers
for col in id_columns:
    df_urls_clean[col] = pd.to_numeric(
        df_urls_clean[col],
        errors="coerce"
    ).astype("Int64")

# Check missing identifiers
missing_ids = df_urls_clean[id_columns].isna().sum()

# Check non-positive identifiers
invalid_ids = (
    df_urls_clean[id_columns]
    .le(0)
    .sum()
)

# Check consistency between URL identifiers
id_mismatches = (
    df_urls_clean["id_ext_url"]
    != df_urls_clean["url_id"]
).fillna(False).sum()

# Check duplicate post-URL relationships
duplicate_relationships = (
    df_urls_clean.duplicated(
        subset=["truth_id", "url_id"]
    ).sum()
)

# Build validation summary
identifier_validation = pd.DataFrame({
    "Missing values": missing_ids,
    "Non-positive values": invalid_ids
})

print("IDENTIFIER VALIDATION")
print("-" * 40)

display(identifier_validation)

print("\nID mismatches:", id_mismatches)

print(
    "Duplicate post-URL relationships:",
    duplicate_relationships
)

print("\nUnique identifiers:")
print(
    df_urls_clean[id_columns].nunique()
)

print("\nDataset shape:")
print(df_urls_clean.shape)

IDENTIFIER VALIDATION
----------------------------------------


,Missing values,Non-positive values
truth_id,0,0
id_ext_url,0,0
url_id,0,0



ID mismatches: 0
Duplicate post-URL relationships: 0

Unique identifiers:
truth_id      247461
id_ext_url    173944
url_id        173944
dtype: int64

Dataset shape:
(252872, 4)


## URL Syntax Validation

We validate the syntax of external URLs to identify
malformed or potentially invalid values.

A URL is considered syntactically valid if it:
- Uses the HTTP or HTTPS protocol.
- Contains a non-empty hostname.
- Does not contain whitespace in the hostname.

Invalid URLs are inspected separately before deciding
whether they can be corrected or should be preserved.

No automatic URL reconstruction is performed.

In [4]:
from urllib.parse import urlsplit
import pandas as pd

# Define URL validation function
def is_valid_url(value):

    if pd.isna(value):
        return False

    if not isinstance(value, str):
        return False

    value = value.strip()

    if not value:
        return False

    try:
        parsed = urlsplit(value)

        return (
            parsed.scheme.lower() in ("http", "https")
            and bool(parsed.hostname)
            and not any(
                char.isspace()
                for char in parsed.hostname
            )
        )

    except ValueError:
        return False


# Validate URL syntax
df_urls_clean["is_valid_url"] = (
    df_urls_clean["url"].apply(is_valid_url)
)

# Count valid and invalid URLs
valid_count = df_urls_clean["is_valid_url"].sum()

invalid_count = (
    ~df_urls_clean["is_valid_url"]
).sum()

print("URL SYNTAX VALIDATION")
print("-" * 40)

print("Valid URLs:", valid_count)
print("Invalid URLs:", invalid_count)

print(
    "Invalid percentage:",
    round(
        invalid_count / len(df_urls_clean) * 100,
        4
    ),
    "%"
)

# Inspect invalid records
print("\nINVALID URL RECORDS")

display(
    df_urls_clean.loc[
        ~df_urls_clean["is_valid_url"],
        [
            "truth_id",
            "id_ext_url",
            "url_id",
            "url"
        ]
    ]
)

URL SYNTAX VALIDATION
----------------------------------------
Valid URLs: 252866
Invalid URLs: 6
Invalid percentage: 0.0024 %

INVALID URL RECORDS


,truth_id,id_ext_url,url_id,url
131989,322988,86769,86769,gracechong
154701,716921,119437,119437,thomaswolfe
155198,717643,119478,119478,PatriotPlymouth2
155694,718755,119561,119561,TexanSeekingTruth
157700,549862,106724,106724,Moon52
185378,856834,140667,140667,DanMeyerswords


## Handling Malformed URLs

The URL validation identified six malformed values
(0.0024% of the dataset), all resembling usernames rather
than valid web addresses.

Since the correct URLs cannot be reconstructed reliably,
these records are excluded from the cleaned dataset.

The excluded records are preserved separately for
traceability and possible future investigation.

In [5]:
# Preserve malformed URL records for documentation
df_invalid_urls = df_urls_clean.loc[
    ~df_urls_clean["is_valid_url"]
].copy()

# Keep only records with syntactically valid URLs
df_urls_clean = df_urls_clean.loc[
    df_urls_clean["is_valid_url"]
].copy()

# Remove temporary validation column
df_urls_clean = df_urls_clean.drop(
    columns=["is_valid_url"]
)

# Validate cleaning results
print("URL CLEANING RESULTS")
print("-" * 40)

print("Original records:", len(df_urls_raw))
print("Exact duplicates removed:", exact_duplicates)
print("Malformed URL records excluded:", len(df_invalid_urls))
print("Final cleaned records:", len(df_urls_clean))

print("\nRemaining exact duplicates:",
      df_urls_clean.duplicated().sum())

print("Remaining malformed URLs:",
      (~df_urls_clean["url"].apply(is_valid_url)).sum())

print("\nFinal dataset shape:")
print(df_urls_clean.shape)

# Verify expected row count
assert len(df_urls_clean) == (
    len(df_urls_raw)
    - exact_duplicates
    - len(df_invalid_urls)
)

assert df_urls_clean["url"].apply(is_valid_url).all()

URL CLEANING RESULTS
----------------------------------------
Original records: 252877
Exact duplicates removed: 5
Malformed URL records excluded: 6
Final cleaned records: 252866

Remaining exact duplicates: 0
Remaining malformed URLs: 0

Final dataset shape:
(252866, 4)


## Referential Integrity with the Main Truths Dataset

The external URLs dataset contains relationships between
Truth Social posts and external web addresses.

The truth_id attribute acts as a foreign key referencing
the id attribute in truths.csv.

We investigate whether all referenced posts are present
in the main dataset.

Unmatched references are preserved because they may
correspond to posts available in additional_truths.arff
or to posts outside the coverage of the main corpus.

No records are removed during this validation.

In [6]:
from pathlib import Path
import pandas as pd

# Define path to the cleaned Truths dataset
TRUTHS_CLEAN_PATH = DATA_DIR / "truths_clean.csv"

if not TRUTHS_CLEAN_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {TRUTHS_CLEAN_PATH}"
    )

# Load only the post identifiers
df_truths_ids = pd.read_csv(
    TRUTHS_CLEAN_PATH,
    usecols=["id"],
    dtype={"id": "Int64"}
)

print("Truths dataset loaded successfully!")
print("Total post IDs:", len(df_truths_ids))
# Load only the post identifiers
df_truths_ids = pd.read_csv(
    TRUTHS_CLEAN_PATH,
    usecols=["id"],
    dtype={"id": "Int64"}
)

# Extract unique post identifiers
valid_truth_ids = pd.Index(
    df_truths_ids["id"].dropna().unique()
)

# Check referential integrity
df_urls_clean["truth_exists_in_main"] = (
    df_urls_clean["truth_id"].isin(valid_truth_ids)
)

# Count matched and unmatched relationships
matched_relations = (
    df_urls_clean["truth_exists_in_main"].sum()
)

unmatched_relations = (
    ~df_urls_clean["truth_exists_in_main"]
).sum()

# Count distinct missing post identifiers
unmatched_truth_ids = (
    df_urls_clean.loc[
        ~df_urls_clean["truth_exists_in_main"],
        "truth_id"
    ].nunique()
)

# Build referential integrity summary
referential_summary = pd.DataFrame({
    "Check": [
        "Total URL relationships",
        "Matched relationships",
        "Unmatched relationships",
        "Distinct unmatched post IDs"
    ],
    "Count": [
        len(df_urls_clean),
        matched_relations,
        unmatched_relations,
        unmatched_truth_ids
    ]
})

print("REFERENTIAL INTEGRITY — MAIN TRUTHS")
print("-" * 45)

display(referential_summary)

print(
    "\nUnmatched percentage:",
    round(
        unmatched_relations / len(df_urls_clean) * 100,
        2
    ),
    "%"
)

print("\nExamples of unmatched relationships:")

display(
    df_urls_clean.loc[
        ~df_urls_clean["truth_exists_in_main"],
        ["truth_id", "id_ext_url", "url_id", "url"]
    ].head(10)
)

Truths dataset loaded successfully!
Total post IDs: 1038821
REFERENTIAL INTEGRITY — MAIN TRUTHS
---------------------------------------------


,Check,Count
0,Total URL relationships,252866
1,Matched relationships,220694
2,Unmatched relationships,32172
3,Distinct unmatched post IDs,30815



Unmatched percentage: 12.72 %

Examples of unmatched relationships:


,truth_id,id_ext_url,url_id,url
1,41909,1780,1780,https://www.supremecourt.gov/opinions/21pdf/19...
34,120,10,10,https://hannity.com/media-room/ben-on-the-mend...
41,125,15,15,https://hannity.com/media-room/jimmy-eat-crow-...
58,136,26,26,https://hannity.com/media-room/constitutional-...
59,494771,26,26,https://hannity.com/media-room/constitutional-...
60,456204,26,26,https://hannity.com/media-room/constitutional-...
61,662908,26,26,https://hannity.com/media-room/constitutional-...
63,138,28,28,https://hannity.com/media-room/rand-slam-watch...
71,143,33,33,https://hannity.com/media-room/dead-wrong-manc...
72,631219,33,33,https://hannity.com/media-room/dead-wrong-manc...


## Referential Integrity with Additional Truths

The main Truths dataset does not contain all posts referenced
by the external URLs dataset.

We investigate whether additional_truths.arff can provide
missing post records and improve referential integrity.

Before performing the comparison, we inspect the available
identifiers to ensure that the datasets use compatible keys.

No unmatched relationships are removed or modified.

In [7]:
from pathlib import Path
import pandas as pd

# Define path to additional dataset
ADDITIONAL_PATH = (
    DATA_DIR.parent
    / "Additional_Data"
    / "additional_truths.arff"
)

if not ADDITIONAL_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {ADDITIONAL_PATH}"
    )

# Inspect ARFF attribute definitions
attributes = []

with open(
    ADDITIONAL_PATH,
    "r",
    encoding="utf-8",
    errors="replace"
) as file:

    for line in file:
        line = line.strip()

        if line.lower() == "@data":
            break

        if line.lower().startswith("@attribute"):
            attributes.append(line)

print("ADDITIONAL TRUTHS — SCHEMA INSPECTION")
print("-" * 45)

print("\nARFF attributes:")

for attribute in attributes:
    print(attribute)

print("\nMain Truths columns:")
print(df_truths_ids.columns.tolist())

print("\nExternal URLs identifier columns:")
print(
    df_urls_clean[
        ["truth_id", "id_ext_url", "url_id"]
    ].columns.tolist()
)

print("\nUnmatched post IDs to investigate:")
print(unmatched_truth_ids)

ADDITIONAL TRUTHS — SCHEMA INSPECTION
---------------------------------------------

ARFF attributes:
@attribute id integer
@attribute timestamp string
@attribute text string
@attribute url string
@attribute reply_count real
@attribute retruth_count real
@attribute like_count real
@attribute external_id string
@attribute author real
@attribute time_scraped string

Main Truths columns:
['id']

External URLs identifier columns:
['truth_id', 'id_ext_url', 'url_id']

Unmatched post IDs to investigate:
30815


## Cross-Dataset ID Matching with Additional Truths

The external URLs dataset contains 30,815 distinct post
identifiers that are absent from the main Truths dataset.

We investigate whether these identifiers appear in
additional_truths.arff.

We compare the internal post identifiers and assess the
overlap between the two datasets.

Matching identifiers are treated as potential references
until their semantic compatibility has been verified.

No records are removed or modified during this analysis.

In [8]:
#%pip install liac-arff

In [10]:
import pandas as pd

# Extract only post identifiers from the ARFF file
additional_ids = []

data_started = False

with open(
    ADDITIONAL_PATH,
    "r",
    encoding="utf-8",
    errors="replace"
) as file:

    for line in file:

        line = line.strip()

        # Identify the beginning of the data section
        if line.lower() == "@data":
            data_started = True
            continue

        if not data_started or not line:
            continue

        # Extract the first attribute (post ID)
        raw_id = line.split(",", 1)[0].strip()

        try:
            additional_ids.append(int(raw_id))
        except ValueError:
            continue

# Create identifier Series
additional_post_ids = pd.Index(
    pd.Series(additional_ids, dtype="Int64")
    .dropna()
    .unique()
)

# Identify missing post IDs from external URLs
missing_post_ids = pd.Index(
    df_urls_clean.loc[
        ~df_urls_clean["truth_exists_in_main"],
        "truth_id"
    ].dropna().unique()
)

# Find potentially recovered post IDs
recovered_post_ids = missing_post_ids.intersection(
    additional_post_ids
)

# Identify IDs still missing
still_missing_ids = missing_post_ids.difference(
    additional_post_ids
)

# Count potentially recovered URL relationships
potentially_recovered_relations = (
    df_urls_clean["truth_id"]
    .isin(recovered_post_ids)
    .sum()
)

# Summarize results
recovery_summary = pd.DataFrame({
    "Check": [
        "Distinct missing post IDs",
        "IDs found in additional Truths",
        "IDs still missing",
        "Potentially recovered URL relationships"
    ],
    "Count": [
        len(missing_post_ids),
        len(recovered_post_ids),
        len(still_missing_ids),
        potentially_recovered_relations
    ]
})

print("CROSS-DATASET POST RECOVERY")
print("-" * 45)

display(recovery_summary)

print("\nPotentially recovered percentage:")

print(
    round(
        len(recovered_post_ids)
        / len(missing_post_ids) * 100,
        2
    ),
    "%"
)

print("\nTotal ARFF records with readable IDs:",
      len(additional_ids))

print("Unique additional post IDs:",
      len(additional_post_ids))

print("\nExamples of potentially recovered IDs:")
print(recovered_post_ids[:20].tolist())

CROSS-DATASET POST RECOVERY
---------------------------------------------


,Check,Count
0,Distinct missing post IDs,30815
1,IDs found in additional Truths,0
2,IDs still missing,30815
3,Potentially recovered URL relationships,0



Potentially recovered percentage:
0.0 %

Total ARFF records with readable IDs: 35222
Unique additional post IDs: 35222

Examples of potentially recovered IDs:
[]


In [11]:
import csv
import pandas as pd

# ARFF attribute order:
# id, timestamp, text, url, reply_count,
# retruth_count, like_count, external_id,
# author, time_scraped

additional_records = []

data_started = False
parse_errors = 0

with open(
    ADDITIONAL_PATH,
    "r",
    encoding="utf-8",
    errors="replace",
    newline=""
) as file:

    for line in file:

        stripped = line.strip()

        if stripped.lower() == "@data":
            data_started = True
            continue

        if not data_started or not stripped:
            continue

        try:
            fields = next(
                csv.reader(
                    [line],
                    delimiter=",",
                    quotechar="'",
                    escapechar="\\"
                )
            )

            if len(fields) != 10:
                parse_errors += 1
                continue

            additional_records.append({
                "id": fields[0],
                "external_id": fields[7],
                "url": fields[3]
            })

        except (csv.Error, ValueError):
            parse_errors += 1

df_additional_ids = pd.DataFrame(
    additional_records
)

# Normalize identifier columns
for col in ["id", "external_id"]:

    df_additional_ids[col] = pd.to_numeric(
        df_additional_ids[col],
        errors="coerce"
    ).astype("Int64")

# Load main Truths identifiers
df_truths_keys = pd.read_csv(
    TRUTHS_CLEAN_PATH,
    usecols=["id", "external_id"],
    low_memory=False
)

for col in ["id", "external_id"]:

    df_truths_keys[col] = pd.to_numeric(
        df_truths_keys[col],
        errors="coerce"
    ).astype("Int64")

# Compare identifier distributions
summary = pd.DataFrame({
    "Dataset / Attribute": [
        "External URLs - truth_id",
        "Main Truths - id",
        "Main Truths - external_id",
        "Additional Truths - id",
        "Additional Truths - external_id"
    ],
    "Minimum": [
        df_urls_clean["truth_id"].min(),
        df_truths_keys["id"].min(),
        df_truths_keys["external_id"].min(),
        df_additional_ids["id"].min(),
        df_additional_ids["external_id"].min()
    ],
    "Maximum": [
        df_urls_clean["truth_id"].max(),
        df_truths_keys["id"].max(),
        df_truths_keys["external_id"].max(),
        df_additional_ids["id"].max(),
        df_additional_ids["external_id"].max()
    ],
    "Unique values": [
        df_urls_clean["truth_id"].nunique(),
        df_truths_keys["id"].nunique(),
        df_truths_keys["external_id"].nunique(),
        df_additional_ids["id"].nunique(),
        df_additional_ids["external_id"].nunique()
    ]
})

print("IDENTIFIER COMPARISON")
print("-" * 45)

display(summary)

print("\nARFF parsing errors:", parse_errors)

print("\nAdditional Truths preview:")
display(df_additional_ids.head(10))

IDENTIFIER COMPARISON
---------------------------------------------


,Dataset / Attribute,Minimum,Maximum,Unique values
0,External URLs - truth_id,21,978127,247455
1,Main Truths - id,18,2383524,1038821
2,Main Truths - external_id,107760498125955216,113392815931039888,892855
3,Additional Truths - id,108211822140637685,117323511508375196,33533
4,Additional Truths - external_id,108211822140637685,117323511508375196,33533



ARFF parsing errors: 1689

Additional Truths preview:


,id,external_id,url
0,117323511508375196,117323511508375196,https://truthsocial.com/@realDonaldTrump/11732...
1,117323148147220269,117323148147220269,https://truthsocial.com/@realDonaldTrump/11732...
2,117322092055919832,117322092055919832,https://truthsocial.com/@realDonaldTrump/11732...
3,117322065119445871,117322065119445871,https://truthsocial.com/@realDonaldTrump/11732...
4,117320581360547091,117320581360547091,https://truthsocial.com/@realDonaldTrump/11732...
5,117320385350427395,117320385350427395,https://truthsocial.com/@realDonaldTrump/11732...
6,117318371569831036,117318371569831036,https://truthsocial.com/@realDonaldTrump/11731...
7,117318368847920040,117318368847920040,https://truthsocial.com/@realDonaldTrump/11731...
8,117318365806497574,117318365806497574,https://truthsocial.com/@realDonaldTrump/11731...
9,117318329260090686,117318329260090686,https://truthsocial.com/@realDonaldTrump/11731...


In [12]:
# Extract valid external IDs from main Truths
main_external_ids = pd.Index(
    df_truths_keys["external_id"]
    .dropna()
    .unique()
)

# Extract available external IDs from additional Truths
additional_external_ids = pd.Index(
    df_additional_ids["external_id"]
    .dropna()
    .unique()
)

# Calculate overlap
shared_external_ids = (
    main_external_ids.intersection(
        additional_external_ids
    )
)

# Identify additional-only external IDs
additional_only_ids = (
    additional_external_ids.difference(
        main_external_ids
    )
)

# Summarize
overlap_summary = pd.DataFrame({
    "Check": [
        "Unique main external IDs",
        "Parsed additional external IDs",
        "Shared external IDs",
        "Additional-only external IDs"
    ],
    "Count": [
        len(main_external_ids),
        len(additional_external_ids),
        len(shared_external_ids),
        len(additional_only_ids)
    ]
})

print("EXTERNAL ID OVERLAP")
print("-" * 40)

display(overlap_summary)

print("\nOverlap percentage:")
print(
    round(
        len(shared_external_ids)
        / len(additional_external_ids) * 100,
        2
    ),
    "%"
)

print("\nExamples of shared external IDs:")
print(shared_external_ids[:10].tolist())

EXTERNAL ID OVERLAP
----------------------------------------


,Check,Count
0,Unique main external IDs,892855
1,Parsed additional external IDs,33533
2,Shared external IDs,4
3,Additional-only external IDs,33529



Overlap percentage:
0.01 %

Examples of shared external IDs:
[109105107965281984, 112622159627271536, 113009752717208400, 113008796172112928]


In [13]:
# Final Data Quality Assessment

# Check missing values
missing_values = df_urls_clean[
    ["truth_id", "id_ext_url", "url_id", "url"]
].isna().sum()

# Check duplicate records
duplicate_records = df_urls_clean[
    ["truth_id", "id_ext_url", "url_id", "url"]
].duplicated().sum()

# Check duplicate post-URL relationships
duplicate_pairs = df_urls_clean.duplicated(
    subset=["truth_id", "url_id"]
).sum()

# Check identifier consistency
id_mismatches = (
    df_urls_clean["id_ext_url"]
    != df_urls_clean["url_id"]
).fillna(False).sum()

# Check invalid identifiers
invalid_identifiers = (
    df_urls_clean[
        ["truth_id", "id_ext_url", "url_id"]
    ].le(0).sum().sum()
)

# Validate URL syntax
invalid_urls = (
    ~df_urls_clean["url"].apply(is_valid_url)
).sum()

# Check unmatched post references
unmatched_references = (
    ~df_urls_clean["truth_exists_in_main"]
).sum()

# Build final quality report
quality_report = pd.DataFrame({
    "Check": [
        "Missing values",
        "Duplicate records",
        "Duplicate post-URL relationships",
        "Inconsistent URL identifiers",
        "Invalid identifiers",
        "Malformed URLs",
        "Unmatched post references"
    ],
    "Count": [
        missing_values.sum(),
        duplicate_records,
        duplicate_pairs,
        id_mismatches,
        invalid_identifiers,
        invalid_urls,
        unmatched_references
    ]
})

print("FINAL DATA QUALITY ASSESSMENT")
print("-" * 45)

display(quality_report)

print("\nDATASET COMPARISON")
print("-" * 45)

print("Original records:", len(df_urls_raw))
print("Exact duplicates removed:", exact_duplicates)
print("Malformed URLs excluded:", len(df_invalid_urls))
print("Final records:", len(df_urls_clean))

print("\nFinal dataset shape:", df_urls_clean.shape)

# Validate cleaning invariants
assert duplicate_records == 0
assert duplicate_pairs == 0
assert invalid_urls == 0
assert id_mismatches == 0
assert invalid_identifiers == 0
assert missing_values.sum() == 0

assert len(df_urls_clean) == (
    len(df_urls_raw)
    - exact_duplicates
    - len(df_invalid_urls)
)

print("\nAll internal data quality checks passed.")

FINAL DATA QUALITY ASSESSMENT
---------------------------------------------


,Check,Count
0,Missing values,0
1,Duplicate records,0
2,Duplicate post-URL relationships,0
3,Inconsistent URL identifiers,0
4,Invalid identifiers,0
5,Malformed URLs,0
6,Unmatched post references,32172



DATASET COMPARISON
---------------------------------------------
Original records: 252877
Exact duplicates removed: 5
Malformed URLs excluded: 6
Final records: 252866

Final dataset shape: (252866, 5)

All internal data quality checks passed.
